# Train the LLaMA style model on TinyStories (Kaggle GPU notebook)

This notebook is self contained. It takes the model from the earlier notebooks (RoPE, RMSNorm, SwiGLU, grouped query attention, KV cache) and trains it on TinyStories, a dataset of short children's stories written in simple language. Small models trained on it produce readable stories, which makes it a good first real dataset.

What it does:

1. Streams the stories from Hugging Face (no full download), filters very short ones, removes duplicates, and keeps the official validation split separate.
2. Trains a byte level BPE tokenizer with a vocabulary of 4096 using the fast tokenizers library, then encodes everything and packs it into one flat file of token ids per split.
3. Trains the model with gradient accumulation, mixed precision (bf16 when the GPU supports it, fp16 with loss scaling otherwise), a warmup plus cosine learning rate schedule, gradient clipping, and evaluation.
4. Logs tokens per second, achieved TFLOPs, MFU (the share of the GPU's peak throughput used) and GPU memory.
5. Saves checkpoints with everything needed to resume exactly, and can stop itself after a time budget.
6. Reports validation loss against simple baselines and prints sample stories.

How to run it on Kaggle:

- Create a new notebook and upload this file (File, Import Notebook).
- In the settings panel, set Accelerator to a GPU (T4 or P100 both work) and turn Internet on. The dataset is downloaded from Hugging Face, so without internet the data step fails.
- Run all cells. The first cells print the settings and run a quick self check of the data pipeline.
- To keep the results, use Save Version and Save and Run All. Files in /kaggle/working become the notebook output.

If the session ends early, start a new session with the previous output attached as an input or saved in /kaggle/working, and run again. The notebook resumes from the last checkpoint.

The time estimates below are guesses, because this notebook could not be run on a GPU while it was written. The training log prints the real tokens per second and an estimated time left after the first few steps. The data pipeline, packing, resume, and both mixed precision paths were tested on a CPU with a small text file by switching SMOKE to True.

In [ ]:
# Settings. Change values here.
SMOKE = False               # True runs the whole pipeline in about a minute on a CPU with data/input.txt, for testing only
DTYPE_OVERRIDE = None       # force "float32", "bfloat16" or "float16"; None picks automatically
DEBUG_STOP_AT = None        # stop after this step, used to test resume
RESUME = True               # continue from the last checkpoint if one exists
COMPILE = False             # torch.compile can speed things up but sometimes fails on older GPUs
TIME_BUDGET_MIN = None      # stop gracefully after this many minutes of training, for example 150

# data
N_TRAIN_STORIES = 300_000   # about 70 million tokens. The full set is about 2.1 million stories
N_VAL_STORIES = 5_000
TOK_TRAIN_STORIES = 100_000 # stories used to train the tokenizer
VOCAB_SIZE = 4096
MIN_CHARS = 100             # stories shorter than this are dropped

# model, about 25 million parameters
MODEL = dict(block_size=512, n_layer=8, n_head=8, n_kv_head=4, n_embd=512, dropout=0.0)

# training
BATCH_SIZE = 32             # sequences per micro batch. Lower it if the GPU runs out of memory
GRAD_ACCUM = 4              # micro batches per optimizer step
TARGET_TOKENS = 150_000_000 # total tokens to train on, roughly two passes over 300k stories
LR, MIN_LR, WARMUP = 1e-3, 1e-4, 200
WEIGHT_DECAY, BETA1, BETA2, GRAD_CLIP = 0.1, 0.9, 0.95, 1.0
EVAL_INTERVAL, EVAL_ITERS, SAVE_INTERVAL, LOG_INTERVAL = 250, 50, 250, 25
SEED = 1337

if SMOKE:
    N_TRAIN_STORIES, N_VAL_STORIES, TOK_TRAIN_STORIES, VOCAB_SIZE, MIN_CHARS = 100_000, 10_000, 100_000, 512, 40
    MODEL = dict(block_size=64, n_layer=2, n_head=4, n_kv_head=2, n_embd=64, dropout=0.0)
    BATCH_SIZE, GRAD_ACCUM, TARGET_TOKENS = 8, 2, 8 * 2 * 64 * 60
    WARMUP, EVAL_INTERVAL, EVAL_ITERS, SAVE_INTERVAL, LOG_INTERVAL = 5, 20, 5, 20, 10

In [ ]:
import os, sys, math, time, json, hashlib, itertools, subprocess, importlib, random
from collections import Counter
from contextlib import nullcontext
from dataclasses import dataclass, replace, asdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

for pkg in ("datasets", "tokenizers"):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

if torch.cuda.is_available():
    device, device_type = "cuda", "cuda"
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    gpu_name = torch.cuda.get_device_name(0)
    cc_major = torch.cuda.get_device_capability(0)[0]
    DTYPE = "bfloat16" if cc_major >= 8 else "float16"      # T4 and P100 have no real bf16 support
elif SMOKE:
    device, device_type, gpu_name, DTYPE = "cpu", "cpu", "cpu", "float32"
else:
    raise RuntimeError("No GPU found. In the Kaggle settings panel, set Accelerator to a GPU and run again.")
if DTYPE_OVERRIDE:
    DTYPE = DTYPE_OVERRIDE

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
OUT_DIR = os.path.join(WORK, "tinystories_smoke" if SMOKE else "tinystories_run")
os.makedirs(OUT_DIR, exist_ok=True)
print(f"device {device} ({gpu_name}) | precision {DTYPE} | torch {torch.__version__} | output folder {OUT_DIR}")

## The model

The same model code as llama_gpt.py from the earlier notebooks: RoPE, RMSNorm, SwiGLU, grouped query attention, tied embeddings, and a KV cache for generation. It was tested for causality, for matching a per head attention reference, and for exact KV cache equivalence in notebook 04.

In [ ]:
"""LLaMA style decoder only transformer (RoPE, RMSNorm, SwiGLU, grouped query attention, KV cache).

The same code as notebook 04, in a module so that train.py, generate.py and the tests can import it.
Config flags also allow the GPT-2 style variant: pos="learned", norm="layer", mlp="gelu", n_kv_head=n_head.
"""
import math
from dataclasses import dataclass, replace, asdict

import torch
import torch.nn as nn
import torch.nn.functional as F


class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d))
        self.eps = eps

    def forward(self, x):
        xf = x.float()
        out = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return out.type_as(x) * self.weight

class LayerNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d))
        self.eps = eps

    def forward(self, x):
        xf = x.float()
        out = (xf - xf.mean(-1, keepdim=True)) / torch.sqrt(xf.var(-1, keepdim=True, unbiased=False) + self.eps)
        return out.type_as(x) * self.weight

def rope_cos_sin(positions, head_dim, theta=10000.0):
    inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2, device=positions.device).float() / head_dim))
    angles = positions.float()[:, None] * inv_freq[None, :]          # (T, head_dim / 2)
    emb = torch.cat([angles, angles], dim=-1)                          # same angle for both members of a pair
    return emb.cos(), emb.sin()

def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)

def apply_rope(x, cos, sin):
    xf = x.float()
    return (xf * cos + rotate_half(xf) * sin).type_as(x)


@dataclass
class Config:
    block_size: int = 128
    vocab_size: int = 65
    n_layer: int = 4
    n_head: int = 4
    n_kv_head: int = 2
    n_embd: int = 128
    pos: str = "rope"          # "rope" or "learned"
    norm: str = "rms"          # "rms" or "layer"
    mlp: str = "swiglu"        # "swiglu" or "gelu"
    dropout: float = 0.0
    rope_theta: float = 10000.0

def make_norm(cfg):
    return RMSNorm(cfg.n_embd) if cfg.norm == "rms" else LayerNorm(cfg.n_embd)

def swiglu_hidden(n_embd, multiple_of=32):
    return multiple_of * ((int(8 * n_embd / 3) + multiple_of - 1) // multiple_of)     # about 8/3 of the width

class Attention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert cfg.n_embd % cfg.n_head == 0 and cfg.n_head % cfg.n_kv_head == 0
        self.cfg = cfg
        self.hd = cfg.n_embd // cfg.n_head
        self.n_rep = cfg.n_head // cfg.n_kv_head
        self.wq = nn.Linear(cfg.n_embd, cfg.n_head * self.hd, bias=False)
        self.wk = nn.Linear(cfg.n_embd, cfg.n_kv_head * self.hd, bias=False)
        self.wv = nn.Linear(cfg.n_embd, cfg.n_kv_head * self.hd, bias=False)
        self.wo = nn.Linear(cfg.n_head * self.hd, cfg.n_embd, bias=False)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x, start_pos=0, cache=None, use_cache=False):
        B, T, _ = x.shape
        cfg = self.cfg
        q = self.wq(x).view(B, T, cfg.n_head, self.hd).transpose(1, 2)
        k = self.wk(x).view(B, T, cfg.n_kv_head, self.hd).transpose(1, 2)
        v = self.wv(x).view(B, T, cfg.n_kv_head, self.hd).transpose(1, 2)
        if cfg.pos == "rope":
            cos, sin = rope_cos_sin(torch.arange(start_pos, start_pos + T, device=x.device), self.hd, cfg.rope_theta)
            q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        if cache is not None:
            k = torch.cat([cache[0], k], dim=2)
            v = torch.cat([cache[1], v], dim=2)
        new_cache = (k, v) if use_cache else None
        S = k.size(2)
        if self.n_rep > 1:                                   # each key/value head serves n_rep query heads
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)
        p = cfg.dropout if self.training else 0.0
        if T == S:
            y = F.scaled_dot_product_attention(q, k, v, dropout_p=p, is_causal=True)
        elif T == 1:
            y = F.scaled_dot_product_attention(q, k, v, dropout_p=p)             # one new token sees everything cached
        else:
            mask = torch.ones(T, S, dtype=torch.bool, device=x.device).tril(diagonal=S - T)
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=p)
        y = y.transpose(1, 2).contiguous().view(B, T, cfg.n_head * self.hd)
        return self.drop(self.wo(y)), new_cache

class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        h = swiglu_hidden(cfg.n_embd)
        self.w1 = nn.Linear(cfg.n_embd, h, bias=False)       # gate
        self.w3 = nn.Linear(cfg.n_embd, h, bias=False)       # value
        self.w2 = nn.Linear(h, cfg.n_embd, bias=False)       # back down
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x):
        return self.drop(self.w2(F.silu(self.w1(x)) * self.w3(x)))

class GeluMLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.w1 = nn.Linear(cfg.n_embd, 4 * cfg.n_embd, bias=False)
        self.w2 = nn.Linear(4 * cfg.n_embd, cfg.n_embd, bias=False)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x):
        return self.drop(self.w2(F.gelu(self.w1(x))))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n1, self.attn = make_norm(cfg), Attention(cfg)
        self.n2, self.mlp = make_norm(cfg), (SwiGLU(cfg) if cfg.mlp == "swiglu" else GeluMLP(cfg))

    def forward(self, x, start_pos=0, cache=None, use_cache=False):
        a, new_cache = self.attn(self.n1(x), start_pos, cache, use_cache)
        x = x + a
        return x + self.mlp(self.n2(x)), new_cache

def sample_next(logits, temperature=1.0, top_k=None):
    logits = logits / max(temperature, 1e-8)
    if top_k is not None:
        kth = torch.topk(logits, min(top_k, logits.size(-1)))[0][:, [-1]]
        logits = logits.masked_fill(logits < kth, float("-inf"))
    return torch.multinomial(F.softmax(logits, dim=-1), 1)

class Model(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.wte = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.wpe = nn.Embedding(cfg.block_size, cfg.n_embd) if cfg.pos == "learned" else None
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.norm = make_norm(cfg)
        self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight
        for m in self.modules():
            if isinstance(m, (nn.Linear, nn.Embedding)):
                nn.init.normal_(m.weight, 0.0, 0.02)
        for n, p in self.named_parameters():
            if n.endswith("wo.weight") or n.endswith("w2.weight"):
                nn.init.normal_(p, 0.0, 0.02 / math.sqrt(2 * cfg.n_layer))

    def num_params(self):
        return sum(p.numel() for p in self.parameters())

    def forward(self, idx, targets=None, kv_caches=None, start_pos=0, use_cache=False):
        B, T = idx.shape
        x = self.wte(idx)
        if self.wpe is not None:
            assert start_pos + T <= self.cfg.block_size, "learned positions cannot go past block_size"
            x = x + self.wpe(torch.arange(start_pos, start_pos + T, device=idx.device))
        x = self.drop(x)
        new_caches = []
        for i, blk in enumerate(self.blocks):
            x, c = blk(x, start_pos, None if kv_caches is None else kv_caches[i], use_cache)
            new_caches.append(c)
        logits = self.lm_head(self.norm(x))
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1)) if targets is not None else None
        return logits, loss, (new_caches if use_cache else None)

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None, use_cache=True):
        was_training = self.training
        self.eval()
        bs = self.cfg.block_size
        if not use_cache:
            for _ in range(max_new_tokens):
                logits, _, _ = self(idx[:, -bs:])
                idx = torch.cat([idx, sample_next(logits[:, -1], temperature, top_k)], dim=1)
        else:
            if self.wpe is not None:
                assert idx.size(1) + max_new_tokens <= bs, "with learned positions the whole sequence must fit in block_size"
            ctx = idx[:, -bs:]
            logits, _, caches = self(ctx, use_cache=True)
            pos = ctx.size(1)
            for _ in range(max_new_tokens):
                nxt = sample_next(logits[:, -1], temperature, top_k)
                idx = torch.cat([idx, nxt], dim=1)
                if caches[0][0].size(2) >= bs:                 # sliding window: keep the newest keys only
                    caches = [(k[:, :, -(bs - 1):], v[:, :, -(bs - 1):]) for k, v in caches]
                logits, _, caches = self(nxt, kv_caches=caches, start_pos=pos, use_cache=True)
                pos += 1
        self.train(was_training)
        return idx


def get_lr(it, lr, min_lr, warmup_iters, max_iters):
    """Linear warmup, then cosine decay to min_lr."""
    if it < warmup_iters:
        return lr * (it + 1) / (warmup_iters + 1)
    if it > max_iters:
        return min_lr
    ratio = (it - warmup_iters) / max(max_iters - warmup_iters, 1)
    return min_lr + 0.5 * (1 + math.cos(math.pi * ratio)) * (lr - min_lr)

## Data: stream, filter, deduplicate

The stories are streamed from Hugging Face, so nothing close to the full dataset has to be stored. A story is dropped if it is shorter than MIN_CHARS characters or if its text (ignoring case and extra whitespace) has been seen before. The validation stories come from the official validation split and go through the same set of seen hashes, so a validation story can never also be a training story.

In [ ]:
def story_iter(split):
    if SMOKE:
        paras = [p.strip() for p in open("data/input.txt", encoding="utf-8").read().split("\n\n") if p.strip()]
        cut = int(0.95 * len(paras))
        return iter(paras[:cut] if split == "train" else paras[cut:])
    from datasets import load_dataset
    try:
        ds = load_dataset("roneneldan/TinyStories", split=split, streaming=True)
        first = next(iter(ds))
        assert "text" in first, f"unexpected columns: {list(first)}"
    except Exception as e:
        raise RuntimeError("Could not read the TinyStories dataset from Hugging Face. On Kaggle, check that Internet is turned on in the "
                           "notebook settings panel, then run again. Original error: " + repr(e))
    return (row["text"] for row in ds)

def text_hash(t):
    return hashlib.blake2b(" ".join(t.split()).lower().encode("utf-8"), digest_size=16).digest()

def clean_and_dedupe(texts, seen, min_chars):
    out, stats = [], Counter()
    for t in texts:
        t = t.strip()
        if len(t) < min_chars:
            stats["too_short"] += 1
            continue
        h = text_hash(t)
        if h in seen:
            stats["duplicate"] += 1
            continue
        seen.add(h)
        out.append(t)
        stats["kept"] += 1
    return out, stats

## Tokenizer and packing

A byte level BPE tokenizer with 4096 tokens is trained on the first TOK_TRAIN_STORIES stories (this uses the fast Rust implementation, the pure Python tokenizer from notebook 02 would take hours at this size). Each story is encoded and followed by an end of text token, then all stories are written one after another into a single file of 16 bit ids. Training samples are random windows of the stream, which wastes no space on padding. A window can cross from one story into the next, which is the usual simple approach.

In [ ]:
EOS_TOKEN = "<|endoftext|>"
TOK_PATH = os.path.join(OUT_DIR, "tokenizer.json")
META_PATH = os.path.join(OUT_DIR, "data_meta.json")
TRAIN_BIN, VAL_BIN = os.path.join(OUT_DIR, "train.bin"), os.path.join(OUT_DIR, "val.bin")

settings_key = dict(smoke=SMOKE, n_train=N_TRAIN_STORIES, n_val=N_VAL_STORIES, tok_train=TOK_TRAIN_STORIES, vocab=VOCAB_SIZE, min_chars=MIN_CHARS)

def build_tokenizer():
    tok = Tokenizer(models.BPE())
    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tok.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(vocab_size=VOCAB_SIZE, special_tokens=[EOS_TOKEN], initial_alphabet=pre_tokenizers.ByteLevel.alphabet(), show_progress=False)
    texts = (t.strip() for t in itertools.islice(story_iter("train"), TOK_TRAIN_STORIES))
    tok.train_from_iterator(texts, trainer)
    tok.save(TOK_PATH)
    return tok

def encode_split(tok, split, n_stories, path, seen):
    eos = tok.token_to_id(EOS_TOKEN)
    total, stats, batch = 0, Counter(), []
    with open(path, "wb") as f:
        def flush():
            nonlocal total
            if not batch:
                return
            clean, st = clean_and_dedupe(batch, seen, MIN_CHARS)
            stats.update(st)
            if clean:
                encs = tok.encode_batch(clean)
                arr = np.fromiter(itertools.chain.from_iterable(e.ids + [eos] for e in encs), dtype=np.uint16)
                f.write(arr.tobytes())
                total += len(arr)
            batch.clear()
        for i, text in enumerate(story_iter(split)):
            if i >= n_stories:
                break
            batch.append(text)
            if len(batch) >= 10_000:
                flush()
        flush()
    return total, dict(stats)

meta = json.load(open(META_PATH)) if os.path.exists(META_PATH) else None
if meta and meta["settings"] == settings_key and all(os.path.exists(p) for p in (TOK_PATH, TRAIN_BIN, VAL_BIN)):
    print("found prepared data, skipping preparation")
    tok = Tokenizer.from_file(TOK_PATH)
else:
    t0 = time.time()
    tok = build_tokenizer()
    print(f"tokenizer trained, vocabulary {tok.get_vocab_size()}, {time.time() - t0:.0f}s")
    seen = set()
    n_train_tok, st_train = encode_split(tok, "train", N_TRAIN_STORIES, TRAIN_BIN, seen)
    n_val_tok, st_val = encode_split(tok, "validation", N_VAL_STORIES, VAL_BIN, seen)
    meta = dict(settings=settings_key, n_train_tokens=n_train_tok, n_val_tokens=n_val_tok, train_stats=st_train, val_stats=st_val)
    json.dump(meta, open(META_PATH, "w"), indent=2)
    print(f"data prepared in {time.time() - t0:.0f}s")

EOS = tok.token_to_id(EOS_TOKEN)
assert tok.get_vocab_size() <= 65535, "ids are stored as 16 bit integers"
print(f"train tokens {meta['n_train_tokens']:,} | val tokens {meta['n_val_tokens']:,} | vocab {tok.get_vocab_size()}")
print("train stories:", meta["train_stats"])
print("validation stories:", meta["val_stats"])

Self check of the data pipeline. These checks run in a few seconds and fail loudly if something in the data is wrong.

In [ ]:
# 1. the filter and the duplicate remover behave as intended
seen_test = set()
out, st = clean_and_dedupe(["x" * 200, "x" * 200, "  " + "X" * 200 + "  ", "short", "y" * 200], seen_test, 100)
assert out == ["x" * 200, "y" * 200] and st == {"kept": 2, "duplicate": 2, "too_short": 1}, (out, st)
print("1. filter and duplicate remover work, including case and whitespace differences")

# 2. the tokenizer is lossless on real stories
sample_stories = [t.strip() for t in itertools.islice(story_iter("train"), 300)]
for s in sample_stories:
    assert tok.decode(tok.encode(s).ids) == s, s[:60]
print(f"2. tokenizer round trip is exact on {len(sample_stories)} stories")

# 3. the packed file has exactly the expected size and starts with the first kept story followed by the end of text token
train_mm = np.memmap(TRAIN_BIN, dtype=np.uint16, mode="r")
val_mm = np.memmap(VAL_BIN, dtype=np.uint16, mode="r")
assert len(train_mm) == meta["n_train_tokens"] and len(val_mm) == meta["n_val_tokens"]
first = next(s for s in sample_stories if len(s) >= MIN_CHARS)
ids = tok.encode(first).ids
assert train_mm[:len(ids)].tolist() == ids and train_mm[len(ids)] == EOS
print("3. packed file size matches the token count and its start matches the first story")

# 4. every id is inside the vocabulary and end of text tokens occur once per story
assert int(train_mm.max()) < tok.get_vocab_size() and int(val_mm.max()) < tok.get_vocab_size()
n_stories_train = meta["train_stats"].get("kept", 0)
assert int((train_mm == EOS).sum()) == n_stories_train, ((train_mm == EOS).sum(), n_stories_train)
print(f"4. all ids are valid and the file holds exactly {n_stories_train:,} stories")

# 5. no validation story appears in the training data, and the training data has no duplicate stories
def stories_from_bin(mm, chunk=10_000):
    eos_pos = np.flatnonzero(np.asarray(mm) == EOS)
    starts = np.concatenate([[0], eos_pos[:-1] + 1])
    for lo in range(0, len(eos_pos), chunk):
        seqs = [mm[a:b].tolist() for a, b in zip(starts[lo:lo + chunk], eos_pos[lo:lo + chunk])]
        yield from tok.decode_batch(seqs)

train_hashes = [text_hash(t) for t in stories_from_bin(train_mm)]
val_hashes = [text_hash(t) for t in stories_from_bin(val_mm)]
assert len(train_hashes) == len(set(train_hashes)) == n_stories_train, "duplicate stories remain in the training data"
assert len(val_hashes) == len(set(val_hashes))
assert not (set(train_hashes) & set(val_hashes)), "a validation story also appears in the training data"
print(f"5. {len(train_hashes):,} training and {len(val_hashes):,} validation stories decoded from the packed files: no duplicates, no overlap")

## Batches and baselines

In [ ]:
class Bin:
    """Random windows from a flat file of token ids."""
    def __init__(self, path):
        self.data = np.memmap(path, dtype=np.uint16, mode="r")
        self.n = len(self.data)

    def batch(self, batch_size, block_size, gen):
        assert self.n > block_size + 1, "not enough tokens for one window"
        ix = torch.randint(self.n - block_size - 1, (batch_size,), generator=gen).numpy()
        window = self.data[ix[:, None] + np.arange(block_size + 1)[None, :]].astype(np.int64)
        chunk = torch.from_numpy(window)
        x, y = chunk[:, :-1].contiguous(), chunk[:, 1:].contiguous()
        if device == "cuda":
            return x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)
        return x, y

train_bin, val_bin = Bin(TRAIN_BIN), Bin(VAL_BIN)
V = tok.get_vocab_size()

# baselines: uniform guessing, and a unigram model that only knows how common each token is
counts = np.bincount(train_bin.data[:20_000_000], minlength=V).astype(np.float64) + 1
p_uni = counts / counts.sum()
val_tokens = np.asarray(val_bin.data[:2_000_000])
unigram_loss = float(-np.log(p_uni[val_tokens]).mean())
print(f"uniform guessing {math.log(V):.3f} | unigram model {unigram_loss:.3f}   (validation loss per token, in nats)")

## Training setup

Utilities for the learning rate, evaluation, checkpoints, and the throughput numbers. FLOPs per token are estimated as 6 times the parameter count (forward and backward matrix multiplications) plus 12 times layers times width times context for attention, the usual convention. MFU divides the achieved FLOPs per second by the GPU's peak for 16 bit math, from the table below, so it is only printed for fp16 and bf16 on GPUs in the table.

In [ ]:
PEAK_FLOPS = [("H100", 989e12), ("A100", 312e12), ("V100", 125e12), ("L4", 121e12), ("T4", 65e12), ("P100", 18.7e12)]

def peak_flops(name):
    for key, val in PEAK_FLOPS:
        if key in name:
            return val
    return None

def flops_per_token(cfg, n_params):
    return 6 * n_params + 12 * cfg.n_layer * cfg.n_embd * cfg.block_size

amp_dtype = {"bfloat16": torch.bfloat16, "float16": torch.float16}.get(DTYPE)
ctx = torch.autocast(device_type=device_type, dtype=amp_dtype) if amp_dtype else nullcontext()
try:
    scaler = torch.amp.GradScaler(device_type, enabled=(DTYPE == "float16"))
except (AttributeError, TypeError):                          # older PyTorch
    scaler = torch.cuda.amp.GradScaler(enabled=(DTYPE == "float16"))

@torch.no_grad()
def evaluate(fwd, cfg):
    fwd.eval()
    g = torch.Generator().manual_seed(123)                    # the same validation windows every time
    out = {}
    for name, b in (("train", train_bin), ("val", val_bin)):
        total = 0.0
        for _ in range(EVAL_ITERS):
            x, y = b.batch(BATCH_SIZE, cfg.block_size, g)
            with ctx:
                total += fwd(x, y)[1].item()
        out[name] = total / EVAL_ITERS
    fwd.train()
    return out

def save_ckpt(path, model, opt, it, gen, best_val, history):
    tmp = path + ".tmp"
    torch.save({"model": model.state_dict(), "optimizer": opt.state_dict(), "scaler": scaler.state_dict(), "iter": it,
                "cfg": asdict(cfg), "gen_state": gen.get_state(), "best_val": best_val, "history": history,
                "dtype": DTYPE}, tmp)
    os.replace(tmp, path)                                       # a crash while saving never corrupts the old checkpoint

cfg = Config(vocab_size=V, **MODEL)
tokens_per_step = BATCH_SIZE * GRAD_ACCUM * cfg.block_size
MAX_ITERS = max(1, math.ceil(TARGET_TOKENS / tokens_per_step))

torch.manual_seed(SEED)
model = Model(cfg).to(device)
n_params = sum(p.numel() for p in model.parameters())
fwd = torch.compile(model) if COMPILE else model
decay = [p for p in model.parameters() if p.dim() >= 2]
no_decay = [p for p in model.parameters() if p.dim() < 2]
use_fused = device == "cuda"
opt = torch.optim.AdamW([{"params": decay, "weight_decay": WEIGHT_DECAY}, {"params": no_decay, "weight_decay": 0.0}],
                        lr=LR, betas=(BETA1, BETA2), **({"fused": True} if use_fused else {}))
gen = torch.Generator().manual_seed(SEED)

print(f"{n_params / 1e6:.2f}M parameters | {tokens_per_step:,} tokens per step | {MAX_ITERS:,} steps | {MAX_ITERS * tokens_per_step / 1e6:.0f}M tokens in total")
print(f"passes over the training data: {MAX_ITERS * tokens_per_step / train_bin.n:.2f}")
fpt = flops_per_token(cfg, n_params)
print(f"{fpt / 1e6:.0f} MFLOPs per token")

## Train

The loop logs every LOG_INTERVAL steps. Checkpoints go to last.pt every SAVE_INTERVAL steps (for resuming) and to best.pt whenever the validation loss improves. If the notebook is stopped and run again, it picks up from last.pt with the same optimizer state, loss scaler state, and data sampler position, so the result matches an uninterrupted run.

In [ ]:
LAST, BEST = os.path.join(OUT_DIR, "last.pt"), os.path.join(OUT_DIR, "best.pt")
history = {"iter": [], "loss": [], "lr": [], "tok_s": [], "eval_iter": [], "eval_train": [], "eval_val": []}
start_it, best_val = 0, float("inf")
if RESUME and os.path.exists(LAST):
    ck = torch.load(LAST, map_location=device, weights_only=False)
    assert ck["cfg"] == asdict(cfg), "the checkpoint was made with a different model configuration"
    model.load_state_dict(ck["model"]); opt.load_state_dict(ck["optimizer"]); scaler.load_state_dict(ck["scaler"])
    gen.set_state(ck["gen_state"].cpu())
    start_it, best_val, history = ck["iter"], ck["best_val"], ck["history"]
    print(f"resumed from step {start_it}, best validation loss so far {best_val:.4f}")

peak = peak_flops(gpu_name) if DTYPE != "float32" else None
model.train()
t_train0 = time.time()
t_last, it_last, completed = time.time(), start_it, False
stop_reason = None

for it in range(start_it, MAX_ITERS + 1):
    lr = get_lr(it, LR, MIN_LR, WARMUP, MAX_ITERS)
    for g_ in opt.param_groups:
        g_["lr"] = lr
    already = bool(history["eval_iter"]) and history["eval_iter"][-1] == it
    if (it % EVAL_INTERVAL == 0 or it == MAX_ITERS) and not already:
        ev = evaluate(fwd, cfg)
        history["eval_iter"].append(it); history["eval_train"].append(ev["train"]); history["eval_val"].append(ev["val"])
        print(f"step {it:6d}/{MAX_ITERS} | train {ev['train']:.4f} | val {ev['val']:.4f} | perplexity {math.exp(ev['val']):.2f}")
        if ev["val"] < best_val:
            best_val = ev["val"]
            save_ckpt(BEST, model, opt, it, gen, best_val, history)
    if it == MAX_ITERS:
        completed = True
        save_ckpt(LAST, model, opt, it, gen, best_val, history)
        break
    if DEBUG_STOP_AT is not None and it >= DEBUG_STOP_AT:
        stop_reason = f"stopped at step {it} (DEBUG_STOP_AT)"
        save_ckpt(LAST, model, opt, it, gen, best_val, history)
        break
    if TIME_BUDGET_MIN is not None and (time.time() - t_train0) / 60 > TIME_BUDGET_MIN:
        stop_reason = f"time budget of {TIME_BUDGET_MIN} minutes used up at step {it}"
        save_ckpt(LAST, model, opt, it, gen, best_val, history)
        break
    if it > start_it and it % SAVE_INTERVAL == 0:
        save_ckpt(LAST, model, opt, it, gen, best_val, history)

    opt.zero_grad(set_to_none=True)
    total = 0.0
    for _ in range(GRAD_ACCUM):
        x, y = train_bin.batch(BATCH_SIZE, cfg.block_size, gen)
        with ctx:
            loss = fwd(x, y)[1] / GRAD_ACCUM
        scaler.scale(loss).backward()
        total += loss.item()
    scaler.unscale_(opt)
    gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    scaler.step(opt)
    scaler.update()

    history["iter"].append(it); history["loss"].append(total); history["lr"].append(lr)
    if it % LOG_INTERVAL == 0 and it > it_last:
        if device == "cuda":
            torch.cuda.synchronize()
        now = time.time()
        tok_s = tokens_per_step * (it - it_last) / (now - t_last)
        history["tok_s"].append((it, tok_s))
        tflops = tok_s * fpt / 1e12
        mfu = f" | MFU {100 * tok_s * fpt / peak:4.1f}%" if peak else ""
        mem = f" | {torch.cuda.max_memory_allocated() / 2**30:.1f} GB" if device == "cuda" else ""
        eta = (MAX_ITERS - it) * tokens_per_step / tok_s / 60
        print(f"   step {it:6d} | loss {total:.4f} | lr {lr:.2e} | gnorm {float(gnorm):.2f} | {tok_s:,.0f} tok/s | {tflops:.1f} TFLOPs{mfu}{mem} | about {eta:.0f} min left")
        t_last, it_last = now, it

if stop_reason:
    print("\n" + stop_reason + ". Run this notebook again to resume from the last checkpoint.")
elif completed:
    print(f"\ntraining finished in {(time.time() - t_train0) / 60:.1f} minutes (this session), best validation loss {best_val:.4f}")

In [ ]:
if len(history["iter"]) > 5:
    fig, ax = plt.subplots(1, 3, figsize=(16, 4))
    k = min(25, max(1, len(history["loss"]) // 5))
    sm = np.convolve(history["loss"], np.ones(k) / k, mode="valid")
    ax[0].plot(history["iter"][k - 1:], sm, label="train (smoothed)")
    ax[0].plot(history["eval_iter"], history["eval_val"], "o-", label="validation")
    ax[0].axhline(unigram_loss, color="gray", linestyle="--", label="unigram baseline")
    ax[0].set_xlabel("step"); ax[0].set_ylabel("loss"); ax[0].legend(); ax[0].set_title("loss")
    ax[1].plot(history["iter"], history["lr"]); ax[1].set_xlabel("step"); ax[1].set_title("learning rate")
    if history["tok_s"]:
        ax[2].plot([a for a, _ in history["tok_s"]], [b for _, b in history["tok_s"]]); ax[2].set_xlabel("step"); ax[2].set_title("tokens per second")
    plt.tight_layout(); plt.show()

## Final evaluation and samples

Reload the best checkpoint into a fresh model, check that it reproduces the saved validation loss, and compare with the baselines. Then generate stories with the KV cache from a few prompts. Generation stops at the end of text token.

In [ ]:
ck = torch.load(BEST, map_location=device, weights_only=False)
best_model = Model(Config(**ck["cfg"])).to(device)
best_model.load_state_dict(ck["model"])
best_model.eval()
ev = evaluate(best_model, best_model.cfg)
print(f"best checkpoint: validation loss {ev['val']:.4f} (saved {ck['best_val']:.4f}) | train loss {ev['train']:.4f} | perplexity {math.exp(ev['val']):.2f}")
assert abs(ev["val"] - ck["best_val"]) < 0.02, "reloaded model does not reproduce the saved validation loss"
print(f"uniform guessing {math.log(V):.3f} | unigram {unigram_loss:.3f} | model {ev['val']:.3f}")
needed = (math.log(V) if SMOKE else unigram_loss) - 0.5
assert ev["val"] < needed, f"the model should clearly beat the {'uniform' if SMOKE else 'unigram'} baseline"

def tell(prompt, max_new_tokens=250, temperature=0.8, top_k=40, seed=0):
    torch.manual_seed(seed)
    ids = tok.encode(prompt).ids
    x = torch.tensor([ids], device=device)
    with ctx:
        out = best_model.generate(x, max_new_tokens, temperature=temperature, top_k=top_k, use_cache=True)[0].tolist()
    if EOS in out[len(ids):]:
        out = out[:len(ids) + out[len(ids):].index(EOS)]
    return tok.decode(out)

for prompt in ("Once upon a time", "Lily and her mom went to the park", "Tom had a red ball"):
    print("-" * 70)
    print(tell(prompt))

In [ ]:
# decoding with and without the KV cache must give identical text
torch.manual_seed(0)
x = torch.tensor([tok.encode("Once upon a time").ids], device=device)
with ctx:
    a = best_model.generate(x, 60, top_k=1, use_cache=True)
    b = best_model.generate(x, 60, top_k=1, use_cache=False)
if DTYPE == "float32":
    assert torch.equal(a, b)
    print("cached and uncached generation are identical")
else:
    print(f"cached and uncached greedy generation agree on {(a == b).float().mean().item() * 100:.0f}% of tokens (16 bit rounding can flip near ties)")

## Summary and files

In [ ]:
print(f"device {gpu_name} | precision {DTYPE}")
print(f"model {n_params / 1e6:.1f}M parameters, {cfg.n_layer} layers, {cfg.n_embd} wide, context {cfg.block_size}, vocabulary {V}")
print(f"data {meta['n_train_tokens'] / 1e6:.1f}M training tokens, {meta['n_val_tokens'] / 1e6:.2f}M validation tokens")
print(f"trained {history['iter'][-1] + 1 if history['iter'] else 0} steps | best validation loss {best_val:.4f} (perplexity {math.exp(best_val):.2f})")
if history["tok_s"]:
    rates = [r for _, r in history["tok_s"]]
    print(f"typical speed {np.median(rates):,.0f} tokens/s")
print("\nfiles to keep:")
for f in sorted(os.listdir(OUT_DIR)):
    size = os.path.getsize(os.path.join(OUT_DIR, f)) / 2**20
    print(f"  {f:20s} {size:9.1f} MB")
print("\nbest.pt is the model to use, tokenizer.json is its tokenizer. train.bin and val.bin can be deleted to save space.")